In [54]:
import pandas as pd

In [55]:
#training data from kaggle
df = pd.read_csv("spotify_millsongdata.csv")

In [56]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [57]:
#checking for null values
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [58]:
df.shape


(57650, 4)

In [59]:
#taking a sample due to gpu overhead
df=df.sample(5000)

In [60]:
df.shape

(5000, 4)

In [61]:
#dropping link column
df=df.drop('link',axis=1).reset_index(drop=True)

In [62]:
df.tail(10)

,artist,song,text
4990,Ramones,Little Bit O'soul,Now when you're feelin' low \r\nAnd the fish ...
4991,Bing Crosby,Personality,When Madam Pompadour was on a ballroom floor ...
4992,Point Of Grace,God Forbid,"The more I know your power, lord \r\nThe more..."
4993,Offspring,We Are One,We are one with ourselves \r\nWe don't give a...
4994,Primus,Arnie,The man he stepped up to the microphone and he...
4995,Rush,Making Memories,There's a time for feelin' as good as we can ...
4996,Deep Purple,The Battle Rages On,"Been so many words, so much to say \r\nWords ..."
4997,The Monkees,"Anytime, Anyplace, Anywhere","CHORUS: \r\nAnytime, anyplace, anywhere, \r\..."
4998,Maroon 5,I Don't Want To Know,"[Verse 1] \r\nWasted the more I think, the mo..."
4999,Beach Boys,Let The Wind Blow,Let the wind blow \r\nLet the grass grow \r\...


Text cleaning /Text Processing

In [63]:
df['text']=df['text'].str.lower().replace(r'^\w\s',' ',regex=True).replace(r'\n',' ',regex=True)

In [64]:
df

,artist,song,text
0,Ariana Grande,Break Free,"[verse 1] \r if you want it, take it \r i sh..."
1,Metallica,Blackened,blackened is the end \r winter it will send ...
2,Moody Blues,It's Up To You,"when the breeze between us calls, \r love com..."
3,Eagles,I Wish You Peace,wish you peace when the cold winds blow \r w...
4,U2,The Electric Co.,"boy, stupid boy \r don't sit at the table \r..."
...,...,...,...
4995,Rush,Making Memories,there's a time for feelin' as good as we can ...
4996,Deep Purple,The Battle Rages On,"been so many words, so much to say \r words a..."
4997,The Monkees,"Anytime, Anyplace, Anywhere","chorus: \r anytime, anyplace, anywhere, \r i..."
4998,Maroon 5,I Don't Want To Know,"[verse 1] \r wasted the more i think, the mor..."


1Tokenizing->2stopwards removal->3 stemming

In [65]:
import nltk
from nltk.stem.porter import PorterStemmer

In [66]:
stemmer=PorterStemmer()

In [67]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ratan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ratan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Not used stopwords till now

In [68]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ratan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [69]:
def token(txt):
    token=nltk.word_tokenize(txt)
    a=[stemmer.stem(w) for w in token]
    return " ".join(a)

In [70]:
token("you are beautiful,beauty,running")

'you are beauti , beauti , run'

tokenizing and stemming actual data

Tutorial did not stored the stemmed data into df['text']

In [71]:
df['text']=df['text'].apply(lambda x: token(x))

In [72]:
df


,artist,song,text
0,Ariana Grande,Break Free,"[ vers 1 ] if you want it , take it i should h..."
1,Metallica,Blackened,blacken is the end winter it will send throw a...
2,Moody Blues,It's Up To You,"when the breez between us call , love come and..."
3,Eagles,I Wish You Peace,wish you peac when the cold wind blow warm by ...
4,U2,The Electric Co.,"boy , stupid boy do n't sit at the tabl until ..."
...,...,...,...
4995,Rush,Making Memories,there 's a time for feelin ' as good as we can...
4996,Deep Purple,The Battle Rages On,"been so mani word , so much to say word are no..."
4997,The Monkees,"Anytime, Anyplace, Anywhere","choru : anytim , anyplac , anywher , i 'll nev..."
4998,Maroon 5,I Don't Want To Know,"[ vers 1 ] wast the more i think , the more i ..."


In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
Tfid=TfidfVectorizer(analyzer='word', stop_words='english')

In [75]:
matrix=Tfid.fit_transform(df['text'])

In [76]:
similar=cosine_similarity(matrix)

In [77]:
similar[0]

array([1.        , 0.01770302, 0.10160971, ..., 0.08489904, 0.1321922 ,
       0.00291257])

In [78]:
similar[1]

array([0.01770302, 1.        , 0.05088339, ..., 0.017211  , 0.00789738,
       0.0084127 ])

In [82]:
df[df['song']=="Blackened"].index[0]

np.int64(1)

Recommender function

In [83]:
def recommender(song_name):
    matches=df[df['song'].str.lower() ==song_name.lower()]
    if matches.empty:
        print(f"Song '{song_name}' not found in the dataset.")
        return []
    idx=matches.index[0]
    distance=sorted(list(enumerate(similar[idx])),reverse=True,key=lambda x:x[1])
    song=[]
    for so_id in distance[1:5]:
        song.append(df.iloc[so_id[0]].song)
    return song

In [88]:
recommender("Making Memories")

["Let's Make A Memory",
 'Am I Losing Your Memory Or Mine?',
 'Memories Of You',
 'Thnks Fr Th Mmrs']

In [89]:
import pickle

In [92]:
pickle.dump(similar, open("similarity","wb"))

In [94]:
pickle.dump(similar, open("similarity.pkl","wb"))

In [93]:
pickle.dump(df, open("df","wb"))

In [95]:
pickle.dump(df, open("df.pkl","wb"))